# 03. Treinamento com MLflow
Aqui o Cientista de Dados treina o modelo de Risco de BACEN consumindo o Feature Store.

In [ ]:
import mlflow
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
import pandas as pd

ENVIRONMENT = "databricks_volume"
BASE_PATH = "/Volumes/workspace/default/raw_data" if ENVIRONMENT == "databricks_volume" else "file:///tmp/mlops"
fs_path = f"{BASE_PATH}/mlops/feature_store"


In [ ]:
print("Carregando base Gold do Feature Store...")
try:
    from databricks.feature_engineering import FeatureEngineeringClient
    fe = FeatureEngineeringClient()
    df_spark = spark.table("workspace.bacen_mlops.call_center_features")
except:
    df_spark = spark.read.parquet(fs_path)

df_pd = df_spark.toPandas()

# Preenchimento de Nulos e Preparação
df_pd.fillna(0, inplace=True)
X = df_pd.drop(columns=["customer_id", "target_bacen"])
y = df_pd["target_bacen"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
print("Iniciando Treinamento e Rastreamento via MLflow...")
mlflow.xgboost.autolog()

with mlflow.start_run(run_name="BACEN_Risk_Model") as run:
    model = xgb.XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42)
    model.fit(X_train, y_train)
    
    preds = model.predict(X_test)
    preds_proba = model.predict_proba(X_test)[:, 1]
    
    auc = roc_auc_score(y_test, preds_proba)
    acc = accuracy_score(y_test, preds)
    
    print(f"Modelo Treinado! AUC: {auc:.4f} | Accuracy: {acc:.4f}")
    
    # Registrando modelo no Model Registry
    mlflow.xgboost.log_model(model, "bacen_xgboost_model", registered_model_name="BacenRiskXGBoost")
